# Notebook 7 — Naive Bayes

Naive Bayes is a fast, simple classifier based on **probability**. 

We'll use our retail dataset for one example, and a small set of text
messages for another (Naive Bayes is very popular for text).

### Setup

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

### Load the Data

Same retail data as before, with the `HighValue` label (1 = above median
order price, 0 = below).

In [3]:
data = pd.read_csv("data.csv", encoding="latin1")
data = data[(data["Quantity"] > 0) & (data["UnitPrice"] > 0)]
data["TotalPrice"] = data["Quantity"] * data["UnitPrice"]
data = data.sample(200, random_state=1).reset_index(drop=True)
median_price = data["TotalPrice"].median()
data["HighValue"] = (data["TotalPrice"] > median_price).astype(int)
data[["Quantity", "UnitPrice", "HighValue"]].head()

,Quantity,UnitPrice,HighValue
0,6,7.95,1
1,2,1.25,0
2,3,1.25,0
3,1,8.47,0
4,2,3.75,0


## 1. Bayes Theorem

**Bayes Theorem** is a formula for updating a probability once you see new
evidence. It combines what you already believed with what the new data
tells you.

$$ P(A|B) = \frac{P(B|A) \times P(A)}{P(B)} $$


## 2. Conditional Probability

**Conditional probability** is the chance of something happening, **given**
that something else has already happened.

Example: the probability that an order is High Value, **given** that its
Unit Price is above $20.

In [4]:
expensive = data[data["UnitPrice"] > 20]
prob_high_value_given_expensive = expensive["HighValue"].mean()
print("P(High Value | Unit Price > 20):", round(prob_high_value_given_expensive, 2))

P(High Value | Unit Price > 20): 1.0


## 3. Naive Bayes

**Naive Bayes** is a classifier that uses Bayes Theorem to calculate the
probability of each class, using all the features together, and picks the
class with the highest probability.

It's popular because it's fast, simple, and works surprisingly well —
especially for text.

## Why is it called "Naive"?

It's called **"Naive"** because that independence assumption is almost
never actually true in real data — features are usually related to each
other in some way. The model naively pretends they aren't, just to make
the probability calculation simple and fast.

The surprising part: even though this assumption is technically wrong,
Naive Bayes still performs well in practice on many real problems,
especially text classification.

## 4. Independence Assumption

To make the math simple, Naive Bayes assumes all features are
**independent** of each other — meaning knowing one feature tells you
nothing about another.

Example: it assumes `Quantity` and `UnitPrice` have no relationship at all,
even though in real life they might.

## 5. Gaussian Naive Bayes

**Gaussian Naive Bayes** is used when features are **continuous numbers**
(like Quantity or Unit Price) 

In [6]:
X = data[["Quantity", "UnitPrice"]]
y = data["HighValue"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
gnb = GaussianNB()
gnb.fit(X_train, y_train)
predictions = gnb.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, predictions), 2))

Accuracy: 0.7


## 6. Text Classification Use Case

Naive Bayes is one of the most common choices for **text classification** —
like spam detection. Text naturally fits its "independence" assumption
reasonably well: each word is treated as its own separate clue.

Let's build a tiny spam-detection example.

In [7]:
messages = [
    "win a free prize now",
    "claim your free lottery winnings",
    "meeting scheduled for tomorrow",
    "please review the attached report",
    "you have won a free gift card",
    "let's catch up over lunch",
    "urgent: claim your reward now",
    "here are the notes from today's meeting",
]
labels = [1, 1, 0, 0, 1, 0, 1, 0]  # 1 = spam, 0 = not spam

## 7. Multinomial Naive Bayes

**Multinomial Naive Bayes** works with **counts** — like how many times
each word appears in a message. It's the standard choice for text
classification.

In [8]:
vectorizer = CountVectorizer()
word_counts = vectorizer.fit_transform(messages)

mnb = MultinomialNB()
mnb.fit(word_counts, labels)

new_message = ["free prize waiting for you"]
new_counts = vectorizer.transform(new_message)

prediction = mnb.predict(new_counts)
print("Prediction (1=spam, 0=not spam):", prediction[0])


Prediction (1=spam, 0=not spam): 1


## 8. Bernoulli Naive Bayes

**Bernoulli Naive Bayes** only cares whether a word appears or not (yes/no),
not how many times. It works well for short texts where word counts don't
matter as much as word presence.

In [9]:
binary_vectorizer = CountVectorizer(binary=True)
binary_counts = binary_vectorizer.fit_transform(messages)
bnb = BernoulliNB()
bnb.fit(binary_counts, labels)
new_binary = binary_vectorizer.transform(new_message)
prediction = bnb.predict(new_binary)
print("Prediction (1=spam, 0=not spam):", prediction[0])

Prediction (1=spam, 0=not spam): 1
